# 👟 ARShoe M1 Fine-Tuning on dataset/shuffled_v4

This notebook fine-tunes the trained **ARShoe M1** model (`outputs/m1_training/checkpoints/best.pth`) on the expanded **`dataset/shuffled_v4`** dataset (1,156 training images, 143 validation images, 146 test images).

### ⚙️ Fine-Tuning Strategy:
- **Pretrained Checkpoint:** Epoch 166 baseline on v3 (clean `_orig_mod.` stripping)
- **Learning Rate:** `2e-4` with Cosine Annealing decay down to `1e-6`
- **Epochs:** 75 epochs
- **Batch Size:** 16
- **Loss Weights:** Heatmap=4.0, PAF=2.0, Class=1.5 (AdaptiveWingLoss + Masked Smooth L1)
- **Output Directory:** `outputs/m1_finetune_v4/` (preserving original checkpoint!)

## Step 1: GPU Verification & Setup

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2: Mount Google Drive (Optional if uploading directly)

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Check Drive root
DRIVE_ROOT = '/content/drive/MyDrive'
print(f"Drive mounted. Available files: {os.listdir(DRIVE_ROOT)[:5]}")

## Step 3: Extract Code and Dataset into /content
Upload `shoes_vto_finetune.zip` and `shuffled_v4.zip` to your Google Drive root or directly to `/content`.

In [ ]:
import os

# 1. Extract shoes_vto
if os.path.exists('/content/drive/MyDrive/shoes_vto_finetune.zip'):
    !unzip -q /content/drive/MyDrive/shoes_vto_finetune.zip -d /content/
    print('✅ Code extracted from Google Drive!')
elif os.path.exists('/content/shoes_vto_finetune.zip'):
    !unzip -q /content/shoes_vto_finetune.zip -d /content/
    print('✅ Code extracted from /content upload!')
elif os.path.exists('/content/shoes_vto'):
    print('✅ /content/shoes_vto already present!')
else:
    print('⚠️ Please upload shoes_vto_finetune.zip to Drive or /content')

# 2. Extract dataset/shuffled_v4
!mkdir -p /content/dataset
if os.path.exists('/content/drive/MyDrive/shuffled_v4.zip'):
    !unzip -q /content/drive/MyDrive/shuffled_v4.zip -d /content/dataset/
    print('✅ Dataset extracted from Google Drive!')
elif os.path.exists('/content/shuffled_v4.zip'):
    !unzip -q /content/shuffled_v4.zip -d /content/dataset/
    print('✅ Dataset extracted from /content upload!')
elif os.path.exists('/content/dataset/shuffled_v4'):
    print('✅ /content/dataset/shuffled_v4 already present!')
else:
    print('⚠️ Please upload shuffled_v4.zip to Drive or /content')

print('\n📁 Files in /content:', os.listdir('/content'))

## Step 4: Install Dependencies

In [ ]:
%cd /content/shoes_vto
!pip install -q -r requirements.txt

## Step 5: (Optional) Pre-Evaluation Baseline on shuffled_v4
Record initial metrics before fine-tuning:

In [ ]:
%cd /content/shoes_vto
!python evaluate_m1.py \
    --checkpoint outputs/m1_training/checkpoints/best.pth \
    --dataset_root ../dataset/shuffled_v4 \
    --output_dir outputs/v4_baseline_eval \
    --num_vis 5

## Step 6: 🚀 Run Fine-Tuning on dataset/shuffled_v4 (Phase 3)

In [ ]:
%cd /content/shoes_vto
!python fine_tune_m1.py \
    --pretrained outputs/m1_training/checkpoints/best.pth \
    --dataset_root ../dataset/shuffled_v4 \
    --output_dir outputs/m1_finetune_v4 \
    --epochs 75 \
    --batch_size 16 \
    --lr 0.0002 \
    --eta_min 0.000001 \
    --num_workers 4

## Step 7: Post-Fine-Tuning Evaluation & Comparative Visualizations
Evaluate the new `best.pth` checkpoint and produce 3-panel overlays:

In [ ]:
%cd /content/shoes_vto
!python evaluate_m1.py \
    --checkpoint outputs/m1_finetune_v4/checkpoints/best.pth \
    --dataset_root ../dataset/shuffled_v4 \
    --output_dir outputs/v4_finetuned_eval \
    --num_vis 10

## Step 8: Backup Fine-Tuned Checkpoints to Google Drive

In [ ]:
!cp outputs/m1_finetune_v4/checkpoints/best.pth /content/drive/MyDrive/best_finetuned_v4.pth
!cp -r outputs/v4_finetuned_eval /content/drive/MyDrive/v4_finetuned_eval
print("✅ Fine-tuned checkpoint and evaluation artifacts saved to Google Drive!")